In [1]:
import pygame
def play_music(name):
    pygame.mixer.init()
    pygame.mixer.music.load(name)
    pygame.mixer.music.play()


pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [ ]:
import psutil
import os

def memory():
    process = psutil.Process(os.getpid())
    print(f"RAM: {process.memory_info().rss / 1024**3:.2f} GB")

memory()

dataset = MyDataset()
memory()

model = MyModel()
memory()

Dataset cleaning

In [5]:
from datasets import load_dataset
import re

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

def clean(example):
    text = example["text"]

    # remove wiki headings
    text = re.sub(r"=+ .*? =+", "", text)
    text = re.sub(r"@.@" , "", text)

    # whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return {"text": text}
dataset = dataset.map(clean)

cleaned_dataset = dataset["train"].filter(
    lambda x: len(x["text"]) > 50
)

In [6]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = cleaned_dataset.map(tokenize_function,batched=True)

Model importing


In [1]:
import torch
import torch.nn as nn
from transformers import BertConfig
from tqdm.auto import tqdm
from DynamicActivationBertModel import MyBertModel

config = BertConfig(
    vocab_size=30522,
    hidden_size=256,
    num_hidden_layers=6,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=256,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

model = MyBertModel(config)
model_type = "Dynamic_activation_bert_model"
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Parameters: 17,704,524


In [11]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 2. Loss Function (CrossEntropy for Masked Language Modeling or Classification)
criterion = nn.CrossEntropyLoss().to(device)

# 3. Optimizer (Weight decay is crucial for Transformers)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

Experimentation Masked Training loop


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup


vocab_size = 30522
mask_token_id = 103  
sample_size = 1000
epochs = 10
MASK_PROB = 0.15
BATCH_SIZE = 4
prev_avg_loss = None

test_test = tokenized_datasets[0:]["input_ids"]

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
total_steps = sample_size * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps*0.1,
    num_training_steps=total_steps*epochs
)

# --- Prepare DataLoader once, outside the loop ---
input_ids = torch.tensor(tokenized_datasets[0:sample_size]["input_ids"])  # (N, seq_len)
dataset = TensorDataset(input_ids)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()

for epoch in range(epochs):
    total_loss = 0

    for (batch,) in dataloader:
        optimizer.zero_grad()

        data = batch.to(device)             # (B, seq_len)s
        labels = data.clone()               # (B, seq_len)

        # --- Masking ---
        probability_matrix = torch.full(labels.shape, MASK_PROB)
        masked_indices = torch.bernoulli(probability_matrix).bool()  # (B, seq_len)
        inputs = data.clone()
        inputs[masked_indices] = mask_token_id                        # (B, seq_len)

        # --- Forward pass ---
        outputs = model(inputs)             # (B, seq_len, vocab_size)

        # --- Fix: index correctly into 3D output ---
        loss = criterion(
            outputs[masked_indices],        # (num_masked, vocab_size)
            labels[masked_indices]          # (num_masked,)
        )

        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(dataloader)
    if prev_avg_loss == None:
        torch.save(model.state_dict(), "model_snapshots/"+model_type+"/"+str(epoch)+"_model.pth")
    else if prev_avg_loss>avg_loss{
        torch.save(model.state_dict(), "model_snapshots/"+model_type+"/"+str(epoch)+"_model.pth")
    }

    print(f"Epoch {epoch} | Total Loss: {total_loss:.4f} | Avg Loss: {avg_loss:.4f}")

# play_music("01 King Gnu - SPECIALZ.flac")

Epoch 0 | Total Loss: 1676.4558 | Avg Loss: 6.7058
Epoch 1 | Total Loss: 1568.0644 | Avg Loss: 6.2723
Epoch 2 | Total Loss: 1489.3837 | Avg Loss: 5.9575
Epoch 3 | Total Loss: 1451.3539 | Avg Loss: 5.8054
Epoch 4 | Total Loss: 1440.4552 | Avg Loss: 5.7618
Epoch 5 | Total Loss: 1436.2930 | Avg Loss: 5.7452
Epoch 6 | Total Loss: 1449.3298 | Avg Loss: 5.7973
Epoch 7 | Total Loss: 1411.7764 | Avg Loss: 5.6471
Epoch 8 | Total Loss: 1422.3573 | Avg Loss: 5.6894
Epoch 9 | Total Loss: 1409.6913 | Avg Loss: 5.6388


Next token prediction 

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup


vocab_size = 30522
mask_token_id = 103  
sample_size = 1000
epochs = 10
MASK_PROB = 0.15
BATCH_SIZE = 4

test_test = tokenized_datasets[0:]["input_ids"]

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
total_steps = sample_size * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps*0.1,
    num_training_steps=total_steps*epochs
)

# --- Prepare DataLoader once, outside the loop ---
input_ids = torch.tensor(tokenized_datasets[0:sample_size]["input_ids"])  # (N, seq_len)
dataset = TensorDataset(input_ids)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()

for epoch in range(epochs):
    total_loss = 0

    for (batch,) in dataloader:

        optimizer.zero_grad()

        data = batch.to(device)             # (B, seq_len)
        labels = data.clone()      
        sentance_length = len(labels)
        # --- Simulating next token prediction ---
        
        for i in range(sentance_length/2,sentance_length,1):
            next_labels = labels[:i+1]
            inputs = labels[:i]
            inputs.append(mask_token_id)
    #     # --- Forward pass ---
            outputs = model(inputs)             # (B, seq_len, vocab_size)

    #     # --- Fix: index correctly into 3D output ---
            loss = criterion(
                outputs[i+1],        # (num_masked, vocab_size)
                next_labels[i+1]          # (num_masked,)
            )

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            scheduler.step()

    avg_loss = total_loss / len(dataloader)
    if prev_avg_loss == None:
        torch.save(model.state_dict(), "model_snapshots/"+model_type+"/"+str(epoch)+"_model.pth")
    else if prev_avg_loss>avg_loss{
        torch.save(model.state_dict(), "model_snapshots/"+model_type+"/"+str(epoch)+"_model.pth")
    }

    print(f"Epoch {epoch} | Total Loss: {total_loss:.4f} | Avg Loss: {avg_loss:.4f}")

# play_music("01 King Gnu - SPECIALZ.flac")

Semantic Training Loop

In [17]:
from datasets import load_dataset
from transformers import BertTokenizer
from torch.utils.data import DataLoader

ds = load_dataset("sentence-transformers/stsb")
criterion =  nn.MSELoss().to(device)


def tokenize_function_text(text):
    return tokenizer(
        text,
        padding="max_length", 
        truncation=True, 
        max_length=128 
    )

In [7]:
test_inputs = []
for i in ds["train"]:
    i["sentence1_tokens"] = tokenize_function_text(i["sentence1"])
    i["sentence2_tokens"] = tokenize_function_text(i["sentence2"])
    test_inputs.append(i)

In [8]:
import torch
import torch.nn.functional as F
import gc
def sentance_embeddings(text):    
    raw_input_ids = text
    input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    output1 = model(input_tensor1,None,True)
    del input_tensor1
    gc.collect()
    return  output1

def cosine_similarity(embeddings,embeddings2):
# embedding shape: [1, 128, 256]
    # embeddings = embeddings.mean(dim=1)   # [1,256]
    # embeddings2 = embeddings2.mean(dim=1)
    print(embeddings.shape)
    sim = F.cosine_similarity(embeddings, embeddings2)
    return sim


In [18]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import math

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
for i in test_inputs:
    loss = 0
    raw_input_ids = i['sentence1_tokens']['input_ids']
    output1 = sentance_embeddings(raw_input_ids)
    output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])
    sim = cosine_similarity(output1,output2)
    
    loss = abs(sim-i['score'])
    pred = torch.tensor(sim).to(device)
    act = torch.tensor(i['score']).to(device)
    print(pred,act)
    losses = criterion(pred, act)
    loss = losses.mean()
    loss.backward()


torch.Size([1, 256])
tensor([0.9994], device='cuda:0') tensor(1., device='cuda:0')


C:\Users\Gaurav B V\AppData\Local\Temp\ipykernel_19968\2991685881.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred = torch.tensor(sim).to(device)
c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

Validation of the Training

In [ ]:
validation_inputs = []
for i in ds["train"]:
    i["sentence1_tokens"] = tokenize_function_text(i["sentence1"])
    i["sentence2_tokens"] = tokenize_function_text(i["sentence2"])
    validation_inputs.append(i)


In [10]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import math

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
loss = 0
for i in validation_inputs:
    raw_input_ids = i['sentence1_tokens']['input_ids']
    output1 = sentance_embeddings(raw_input_ids)
    output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])

    # input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    print(output2.shape)
    # output1 = model(input_tensor1,None,True)
    sim = cosine_similarity(output1,output2)
    print(sim,i['score'])
    
    loss+= abs(sim-i['score'])
    print(output2.shape)
    print(sim.item())


torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9926], device='mps:0', grad_fn=<SumBackward1>) 1.0
torch.Size([1, 256])
0.9926233291625977
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9978], device='mps:0', grad_fn=<SumBackward1>) 0.76
torch.Size([1, 256])
0.9978430271148682
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9934], device='mps:0', grad_fn=<SumBackward1>) 0.76
torch.Size([1, 256])
0.9933737516403198
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9984], device='mps:0', grad_fn=<SumBackward1>) 0.52
torch.Size([1, 256])
0.9983895421028137
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9956], device='mps:0', grad_fn=<SumBackward1>) 0.85
torch.Size([1, 256])
0.9956103563308716
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9976], device='mps:0', grad_fn=<SumBackward1>) 0.85
torch.Size([1, 256])
0.9975963234901428
torch.Size([1, 256])
torch.Size([1, 256])
tensor([0.9979], device='mps:0', grad_fn=<SumBackward1>) 0.1
torch.Size([1, 256])
0.99793493747

KeyboardInterrupt: 

In [9]:
raw_input_ids = i['sentence1_tokens']['input_ids']
output1 = sentance_embeddings(raw_input_ids)
output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])
print(output1.shape)
print(output2.shape)

torch.Size([1, 256])
torch.Size([1, 256])


In [10]:
raw_input_ids = i['sentence1_tokens']['input_ids']
output1 = sentance_embeddings(raw_input_ids)
output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])
print(output1)
print(output2)

tensor([[ 1.6016, -0.2144, -0.0710,  1.2778, -2.5050, -0.0835, -1.9429, -2.5132,
         -0.2126, -0.0933,  0.3304, -1.0098, -0.2533,  1.3692, -1.2755, -0.0140,
         -2.4571, -1.4669,  2.0046, -1.3845,  2.8056, -0.4272,  2.0125,  1.4370,
          1.8119, -0.4767,  1.6130,  1.1831, -0.8691,  0.6550, -2.7352,  3.2976,
         -0.4638,  0.3704, -0.3266, -1.6200,  2.2034,  0.4513, -0.0453, -0.0389,
          0.2238,  1.9151,  1.8654,  1.3397,  1.4958,  1.7151, -1.2569,  2.0571,
          1.8618,  1.7713, -1.5884,  2.4899, -0.9836, -0.4051, -4.7502,  0.8478,
         -0.7616, -1.9440, -0.8611,  0.5420,  0.7207, -0.4808,  0.4157,  1.0835,
          1.8168, -1.0882, -1.4959, -0.0873,  0.9759,  2.0280, -2.3955, -5.3678,
          0.5921,  0.4895,  0.7599,  1.5041,  3.5452,  1.4657,  0.7081, -0.5743,
         -0.0619,  1.2181,  1.8440,  6.7772,  2.8191, -1.0179, -2.3532, -1.0195,
          3.6286, -4.8502, -0.2055,  0.9838,  1.6472,  0.2302,  2.3480, -0.7083,
         -0.1987, -1.2797, -

In [21]:
import torch 
emb1 = torch.tensor([[ 1.2348,  0.2386,  0.2800,  0.9496, -1.4813, -1.0805, -2.2542, -2.3571,
         -0.1564, -0.5397,  0.3665, -1.2065, -0.2421,  0.0581, -0.3566,  0.2863,
         -2.1741, -1.4072,  1.4715, -1.2135,  3.2552,  0.0144,  1.4374,  0.5465,
          2.0238, -0.7812,  1.7034,  1.2138, -1.9390,  1.0825, -1.9740,  3.9195,
          0.0770,  1.8139, -1.3559, -0.7920,  2.2564,  0.3364, -1.2293,  0.3971,
         -0.0516,  1.8795,  2.0271,  0.9119,  1.0684,  1.6527, -0.5378,  1.8452,
          1.5120,  1.7783, -1.6371,  1.8211, -0.9406, -0.1081, -2.7461,  0.4550,
         -0.6082, -2.2510, -1.1194, -0.0854,  0.0726, -1.3341,  0.3663,  1.4341,
          1.0280, -1.7972, -1.7895,  0.6574,  1.4064,  2.9138, -2.5457, -5.8128,
          0.6951,  0.3528,  0.3521,  1.0871,  3.0419,  1.1403,  1.5479, -0.1763,
         -1.3406,  0.8202,  2.2308,  3.0783,  2.3264, -0.7892, -2.6039,  0.0969,
          4.5042, -4.1561,  1.1078,  0.6670,  0.8057, -0.8638,  2.6479, -0.9154,
          1.1078, -1.7181, -1.9977, -2.9163, -0.7973,  2.5372,  0.9099,  1.3102,
         -1.7055,  2.0357,  0.5844,  1.4695, -0.6545,  2.5721, -2.6509, -0.2280,
         -1.3623, -0.1216,  0.3701,  3.3823,  1.3459,  0.5962,  1.9750, -0.8230,
          1.3762, -0.0806,  1.5067, -1.4300,  0.6598,  0.0841, -0.9707, -0.1296,
          0.3033, -0.7551,  1.4591,  2.8031, -2.3270,  0.0246,  2.2466,  0.7713,
          0.8239, -1.0540, -2.0838,  0.0927,  2.3450,  4.9283, -1.6810,  0.2567,
          1.6633, -1.1197, -0.0909, -4.0505, -1.6124,  2.3180,  2.7203,  0.2096,
         -2.1827, -0.9084, -3.2716,  0.1693,  1.0441, -1.5260,  2.1376,  0.0725,
          2.2704,  0.5637,  0.9229,  1.1503, -3.3290, -0.5577,  1.3899, -1.3229,
         -1.3468, -1.4334,  0.7398,  0.8661, -1.2183, -0.8454,  1.9852, -2.0030,
         -2.0691, -0.2419,  1.2651, -3.7448, -1.6883, -1.3170, -1.1608, -1.1595,
          1.4316, -0.9537,  0.1278,  0.0406, -1.6749,  1.4247,  1.0354, -1.6049,
         -1.0891,  1.7453,  0.4860, -0.8033,  0.5708, -2.8396,  0.3456, -2.0761,
         -0.2593,  2.8028, -1.0782,  0.8027,  2.3332, -1.0229,  0.1623, -1.6918,
         -0.4044, -1.5690, -2.0887,  2.5530, -0.9202,  1.5341,  0.5036, -2.1450,
         -0.3187, -1.4194, -3.4533, -0.2013,  0.5296, -1.1409, -1.3858, -0.2508,
          1.4090,  0.2632, -1.2670, -3.7417, -0.6520, -1.1110,  1.5962, -2.8813,
          1.4789, -0.2699, -2.9905, -0.2307,  0.1764, -1.0848,  1.8412, -0.4293,
          1.8421,  0.0325,  2.4136, -1.7119,  0.9146,  1.3261, -1.8936,  0.6255,
          0.6582,  1.4221, -3.9001, -1.7055, -1.1199,  1.2035, -0.9521,  0.6102]]) 

In [22]:
emb2 = torch.tensor([[ 1.6016, -0.2144, -0.0710,  1.2778, -2.5050, -0.0835, -1.9429, -2.5132,
         -0.2126, -0.0933,  0.3304, -1.0098, -0.2533,  1.3692, -1.2755, -0.0140,
         -2.4571, -1.4669,  2.0046, -1.3845,  2.8056, -0.4272,  2.0125,  1.4370,
          1.8119, -0.4767,  1.6130,  1.1831, -0.8691,  0.6550, -2.7352,  3.2976,
         -0.4638,  0.3704, -0.3266, -1.6200,  2.2034,  0.4513, -0.0453, -0.0389,
          0.2238,  1.9151,  1.8654,  1.3397,  1.4958,  1.7151, -1.2569,  2.0571,
          1.8618,  1.7713, -1.5884,  2.4899, -0.9836, -0.4051, -4.7502,  0.8478,
         -0.7616, -1.9440, -0.8611,  0.5420,  0.7207, -0.4808,  0.4157,  1.0835,
          1.8168, -1.0882, -1.4959, -0.0873,  0.9759,  2.0280, -2.3955, -5.3678,
          0.5921,  0.4895,  0.7599,  1.5041,  3.5452,  1.4657,  0.7081, -0.5743,
         -0.0619,  1.2181,  1.8440,  6.7772,  2.8191, -1.0179, -2.3532, -1.0195,
          3.6286, -4.8502, -0.2055,  0.9838,  1.6472,  0.2302,  2.3480, -0.7083,
         -0.1987, -1.2797, -4.4299, -2.5390, -0.3199,  2.8575, -0.5735,  1.3891,
         -2.1803,  2.9218,  0.6285,  2.2691, -0.4308,  1.1215, -2.1614,  0.6812,
         -1.5196,  0.5472,  0.6371,  3.5658,  1.2836,  0.1506,  2.2566, -0.7243,
          0.1533,  0.4744,  1.3468, -1.7884,  1.0233,  0.4614, -1.3633,  1.1369,
         -0.3384, -0.3666,  1.5436,  2.3318, -2.7017, -0.1162,  2.1828, -0.4184,
          2.2562, -0.9714, -2.4811,  0.1478,  2.3558,  4.3746, -1.5564,  0.0770,
          2.0998, -0.9166, -0.0447, -3.3446, -0.9257,  2.5749,  1.9134,  0.3069,
         -2.6432, -1.7012, -2.9306,  0.4783,  1.2049, -0.9613,  2.3712,  0.1163,
          2.6930,  0.4604,  1.6778,  1.4177, -3.5625, -0.0719,  1.5155, -0.7426,
         -1.8981, -0.9941,  0.6000,  1.1849, -1.4138, -1.2819,  2.1547, -1.6901,
         -1.9180, -0.8334,  0.9352, -3.2119, -1.8043, -1.6480, -0.8397, -0.5817,
          0.7498, -1.4758,  0.2802, -0.4301, -1.1805,  1.5383,  0.8700, -1.4568,
         -0.7221,  1.3859, -0.5122, -0.8254,  0.1465, -1.7847,  0.4736, -1.6745,
         -0.4149,  2.5711, -1.0855,  0.4028,  1.6772, -0.4603,  0.1711, -2.3449,
          0.1024, -1.8294, -1.3844,  2.3283, -0.9012,  0.7179,  0.1567, -2.0015,
         -1.1147, -0.9764, -2.8028, -0.9525,  0.5943, -1.0235, -1.8838,  0.5460,
          1.6898, -1.3650, -2.2851, -3.9734, -1.1985, -1.4439, -0.9606, -1.6387,
          1.4503, -1.4083, -2.3986, -0.8658, -0.7358, -0.8368,  1.9976, -0.8139,
          1.6687,  0.1296,  2.3470, -1.5034,  0.8059,  2.3029, -0.6251,  0.4890,
          0.2468,  1.3614, -3.4207, -2.2415, -1.1773,  0.6214, -1.2693,  0.7035]])

In [25]:
import torch.nn.functional as F

e1 = emb1.squeeze()
e2 = emb2.squeeze()

print("Cosine:", F.cosine_similarity(e1, e2, dim=0))

print("L2 distance:", torch.norm(e1 - e2))

print("Norm1:", torch.norm(e1))
print("Norm2:", torch.norm(e2))

Cosine: tensor(0.9257)
L2 distance: tensor(10.5703)
Norm1: tensor(26.9719)
Norm2: tensor(27.7452)


In [10]:
emb1 = F.normalize(emb1, dim=1)

sim = emb1 @ emb1.T

print(sim.min())
print(sim.max())
print(sim.mean())

tensor(1.)
tensor(1.)
tensor(1.)


In [6]:
mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
off_diag = sim[mask]

print("Off-diagonal min:", off_diag.min().item())
print("Off-diagonal max:", off_diag.max().item())
print("Off-diagonal mean:", off_diag.mean().item())

NameError: name 'sim' is not defined

In [ ]:
F.cosine_similarity(e1, e2, dim=1)

In [4]:
!pip install --upgrade pip

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Users\Gaurav B V\anaconda3\envs\bot\python.exe -m pip install --upgrade pip


In [7]:
!pip install google

In [ ]:
!pip uninstall google
!pip install -U google-genai

^C


In [1]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="Search for 'Gemini API' on Google.",
    tools=[{"type": "computer_use", "environment": "browser"}]
)

print(interaction)

ImportError: cannot import name 'genai' from 'google' (unknown location)